In [1]:
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn

In [2]:
class SpatioTemporalGNN(nn.Module):
    def __init__(self,nodes,input_features,lookback,hidden_dims):
        super(SpatioTemporalGNN,self).__init__()
        self.nodes=nodes
        self.lookback=lookback
        self.hidden_dims=hidden_dims

        self.gcn_weight=nn.Parameter(torch.FloatTensor(input_features,hidden_dims))
        nn.init.xavier_uniform_(self.gcn_weight)

        self.lstm=nn.LSTM(input_size=hidden_dims,hidden_size=hidden_dims,num_layers=1,batch_first=True)
        self.fc=nn.Linear(hidden_dims*lookback,1)

    def forward(self, x, adj):
        # Input 'x' structural footprint: [Batch, Features, Nodes, Timesteps]
        batch_size, _, num_nodes, timesteps = x.shape
        A = torch.tensor(adj, dtype=torch.float32, device=x.device)
        
        # 1. Spatial Processing Feature Loop
        spatial_features = []
        for t in range(timesteps):
            xt = x[:, :, :, t].permute(0, 2, 1)  # Shape becomes: [Batch, Nodes, Features]
            spatial_kernel = torch.matmul(A, xt)  # Aggregate spatial neighbors
            z = torch.matmul(spatial_kernel, self.gcn_weight)  # Map features via learnable weights [B, N, H]
            spatial_features.append(z.unsqueeze(3))
            
        spatial_out = torch.cat(spatial_features, dim=3)  # Recompiled output dimensions: [B, N, H, T]

        # 2. Temporal Sequencer Loop
        outputs = []
        for n in range(num_nodes):
            node_seq = spatial_out[:, n, :, :].permute(0, 2, 1)  # Extract per-node sequences [Batch, Timesteps, Hidden]
            lstm_out, _ = self.lstm(node_seq)
            lstm_flat = lstm_out.reshape(batch_size, -1)
            pred = self.fc(lstm_flat)  # Map down to single future vector element
            outputs.append(pred)
            
        return torch.cat(outputs, dim=1)  # Consolidated shape output: [Batch, Nodes]

In [3]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tensors=torch.load('/kaggle/input/datasets/anubhashukla/gnn-training/gnn_tensors (1).pt',map_location=device)
adj=np.load('/kaggle/input/datasets/anubhashukla/weight-matrix-np/spatial_weight_matrix (1).npy')

In [4]:
x_train,y_train=tensors['x_train'].to(device),tensors['y_train'].to(device)

In [5]:
model=SpatioTemporalGNN(nodes=30, input_features=1, lookback=12, hidden_dims=16).to(device)
optimizer=optim.Adam(model.parameters(),lr=0.01)
criterion=nn.MSELoss()

In [8]:
epochs=30
batch_size=64
model.train()

for epoch in range(epochs):
    total_loss=0
    permutation=torch.randperm(x_train.size(0))
    for i in range(0, x_train.size(0), batch_size):
        optimizer.zero_grad()
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = x_train[indices], y_train[indices]
        
        preds = model(batch_x, adj)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(indices)
        
    print(f"Epoch {epoch+1:03d}/{epochs} | Run Average Mean Squared Error (MSE): {total_loss / x_train.size(0):.4f}")

torch.save(model.state_dict(),'stgnn_model.pth')

Epoch 001/30 | Run Average Mean Squared Error (MSE): 0.0176
Epoch 002/30 | Run Average Mean Squared Error (MSE): 0.0130
Epoch 003/30 | Run Average Mean Squared Error (MSE): 0.0111
Epoch 004/30 | Run Average Mean Squared Error (MSE): 0.0107
Epoch 005/30 | Run Average Mean Squared Error (MSE): 0.0092
Epoch 006/30 | Run Average Mean Squared Error (MSE): 0.0084
Epoch 007/30 | Run Average Mean Squared Error (MSE): 0.0080
Epoch 008/30 | Run Average Mean Squared Error (MSE): 0.0075
Epoch 009/30 | Run Average Mean Squared Error (MSE): 0.0075
Epoch 010/30 | Run Average Mean Squared Error (MSE): 0.0077
Epoch 011/30 | Run Average Mean Squared Error (MSE): 0.0071
Epoch 012/30 | Run Average Mean Squared Error (MSE): 0.0071
Epoch 013/30 | Run Average Mean Squared Error (MSE): 0.0070
Epoch 014/30 | Run Average Mean Squared Error (MSE): 0.0070
Epoch 015/30 | Run Average Mean Squared Error (MSE): 0.0069
Epoch 016/30 | Run Average Mean Squared Error (MSE): 0.0070
Epoch 017/30 | Run Average Mean Squared 